In [6]:
import networkx as nx
import numpy as np
from gwgraphs import *

In [7]:
def read_graph_data():
    # Read the adjacency matrix (edges)
    with open("COX2_A.txt", "r") as f:
        edges = [tuple(map(int, line.split(','))) for line in f]

    # Read graph indicators
    with open("COX2_graph_indicator.txt", "r") as f:
        graph_indicators = [int(line.strip()) for line in f if line.strip()]

    # Read graph labels
    with open("COX2_graph_labels.txt", "r") as f:
        graph_labels = [int(line.strip()) for line in f]

    # Read node labels
    with open("COX2_node_labels.txt", "r") as f:
        node_labels = [int(line.strip()) for line in f]

    # Read node attributes
    with open("COX2_node_attributes.txt", "r") as f:
        node_attributes = [
            list(map(float, line.strip().split(','))) for line in f if line.strip()
        ]

    return edges, graph_indicators, graph_labels, node_labels, node_attributes


def construct_networkx_graph(edges, graph_indicators, graph_labels,
                             node_labels, node_attributes):
    graphs = {}
    graphs_normalized = {}
    
    # Normalize all node attributes
    normalized_attributes = normalize_features(node_attributes)
    
    for node_id, graph_id in enumerate(graph_indicators, start=1):
        if graph_id not in graphs:
            graphs[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])
            graphs_normalized[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])

    for edge in edges:
        node1, node2 = edge
        graph_id = graph_indicators[
            node1 - 1]  # Determine which graph the edge belongs to
        graphs[graph_id].add_edge(node1, node2)
        graphs_normalized[graph_id].add_edge(node1, node2)

    for node_id, (graph_id, node_label,
                  node_attr, norm_attr) in enumerate(zip(graph_indicators, node_labels,
                                              node_attributes, normalized_attributes),
                                          start=1):
        # Original graphs with original features
        graphs[graph_id].nodes[node_id]['label'] = node_label
        graphs[graph_id].nodes[node_id]['attributes'] = node_attr
        
        # Normalized graphs with normalized features
        graphs_normalized[graph_id].nodes[node_id]['label'] = node_label
        graphs_normalized[graph_id].nodes[node_id]['attributes'] = norm_attr

    return graphs, graphs_normalized


def compute_cox_distance_matrix(graph):
    num_nodes = len(graph.nodes)
    distance_matrix = np.full((num_nodes, num_nodes), 10000.0)

    # Get the correct node indices to be used for indexing the matrix
    node_list = list(graph.nodes())
    node_index = {node: i for i, node in enumerate(node_list)}

    for node in graph.nodes:
        lengths = nx.shortest_path_length(graph, source=node, weight="weight")

        for target, length in lengths.items():
            if target in node_index:
                distance_matrix[node_index[node], node_index[target]] = length

    return distance_matrix


def normalize_features(features):
    features = np.array(features)
    # Normalize each feature dimension independently
    mean = np.mean(features, axis=0)
    std = np.std(features, axis=0)
    # Avoid division by zero
    std[std == 0] = 1
    normalized_features = (features - mean) / std
    return normalized_features.tolist()

In [8]:
edges, graph_indicators, graph_labels, node_labels, node_attributes = read_graph_data()
graphs, graphs_normalized = construct_networkx_graph(edges, graph_indicators, graph_labels,
                                                   node_labels, node_attributes)
for graph_id, graph in graphs.items():
    print(
        f"Graph ID: {graph_id}, number of nodes: {graph.number_of_nodes()}, label: {graph.graph['label']}"
        )

Graph ID: 1, number of nodes: 39, label: -1
Graph ID: 2, number of nodes: 36, label: -1
Graph ID: 3, number of nodes: 36, label: 1
Graph ID: 4, number of nodes: 39, label: -1
Graph ID: 5, number of nodes: 42, label: -1
Graph ID: 6, number of nodes: 42, label: -1
Graph ID: 7, number of nodes: 44, label: -1
Graph ID: 8, number of nodes: 39, label: -1
Graph ID: 9, number of nodes: 42, label: 1
Graph ID: 10, number of nodes: 42, label: 1
Graph ID: 11, number of nodes: 45, label: 1
Graph ID: 12, number of nodes: 49, label: 1
Graph ID: 13, number of nodes: 42, label: 1
Graph ID: 14, number of nodes: 35, label: -1
Graph ID: 15, number of nodes: 44, label: 1
Graph ID: 16, number of nodes: 47, label: 1
Graph ID: 17, number of nodes: 45, label: 1
Graph ID: 18, number of nodes: 39, label: 1
Graph ID: 19, number of nodes: 44, label: -1
Graph ID: 20, number of nodes: 44, label: -1
Graph ID: 21, number of nodes: 51, label: -1
Graph ID: 22, number of nodes: 45, label: -1
Graph ID: 23, number of nodes

In [9]:
graphs_list = []
labels_list = []
for graph_id, graph in graphs.items():
    # Add the graph object to the list
    graphs_list.append(graph)

    # Extract the label and add to the label list
    label = graph.graph['label']
    labels_list.append(label)

# Convert graphs into pairwise distance matrices
distance_matrices = [compute_cox_distance_matrix(graph) for graph in graphs_list]

In [13]:
"""fGW-kNN: alpha hyperparameter tuning"""

'fGW-kNN: alpha hyperparameter tuning'

In [14]:
k_list = [1,2,3,4,5,6,7,8,9,10]
num_folds = 10

In [15]:
alpha_list = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
alpha_tests_k = {
        alph: {}
        for alph in alpha_list
    }  # dictionary to store best k information for each alpha

for alpha in alpha_list:
    print()
    print("Currently testing alpha =", alpha)
    print()
    class_acc_dict, by_class_acc_dict, f1_score_dict, best_k_info = fgw_knn_alpha(distance_matrices, graphs_list, labels_list, num_folds, k_list=k_list, method="stratk",alpha=alpha)
    
    # Store the best k information for this alpha
    alpha_tests_k[alpha] = best_k_info


Currently testing alpha = 0.0

Stratified k-fold CV selected
Fold 0:
k=1, Class 1 Accuracy: 30.0%, Fold 0
k=1, Class -1 Accuracy: 78.37837837837837%, Fold 0
k=1, Overall Classification Accuracy: 68.08510638297872%, Fold 0
k=1, F1 Score (weighted avg): 68.63%, Fold 0
k=2, Class 1 Accuracy: 30.0%, Fold 0
k=2, Class -1 Accuracy: 78.37837837837837%, Fold 0
k=2, Overall Classification Accuracy: 68.08510638297872%, Fold 0
k=2, F1 Score (weighted avg): 68.63%, Fold 0
k=3, Class 1 Accuracy: 10.0%, Fold 0
k=3, Class -1 Accuracy: 83.78378378378379%, Fold 0
k=3, Overall Classification Accuracy: 68.08510638297872%, Fold 0
k=3, F1 Score (weighted avg): 65.89%, Fold 0
k=4, Class 1 Accuracy: 10.0%, Fold 0
k=4, Class -1 Accuracy: 83.78378378378379%, Fold 0
k=4, Overall Classification Accuracy: 68.08510638297872%, Fold 0
k=4, F1 Score (weighted avg): 65.89%, Fold 0
k=5, Class 1 Accuracy: 10.0%, Fold 0
k=5, Class -1 Accuracy: 91.8918918918919%, Fold 0
k=5, Overall Classification Accuracy: 74.4680851063

In [16]:
# Find best alpha based on the best k performance for each alpha
best_alpha = alpha_list[0]
best_alpha_acc = 0
best_alpha_f1 = 0

print("\n" + "="*60)
print("SUMMARY OF BEST K PERFORMANCE FOR EACH ALPHA:")
print("="*60)

for alph in alpha_list:
    best_k_info = alpha_tests_k[alph]
    
    print(f"\nalpha={alph}:")
    print(f"  Best k: {best_k_info['best_k']}")
    print(f"  Overall Accuracy: {best_k_info['best_overall_accuracy']:.2f}%")
    print(f"  F1 Score (weighted avg): {best_k_info['best_f1_score']:.2f}%")
    print(f"  Class 1 Accuracy: {best_k_info['best_class_1_accuracy']:.2f}%")
    print(f"  Class -1 Accuracy: {best_k_info['best_class_minus1_accuracy']:.2f}%")

    # Check if this alpha gives the best overall accuracy
    if best_k_info['best_overall_accuracy'] > best_alpha_acc:
        best_alpha_acc = best_k_info['best_overall_accuracy']
        best_alpha = alph

    # Track best F1 score
    if best_k_info['best_f1_score'] > best_alpha_f1:
        best_alpha_f1 = best_k_info['best_f1_score']

print("\n" + "="*60)
print("OVERALL BEST PERFORMANCE:")
print("="*60)
best_info = alpha_tests_k[best_alpha]
print(f"Best alpha: {best_alpha}")
print(f"Best k for this alpha: {best_info['best_k']}")
print(f"Overall Accuracy: {best_info['best_overall_accuracy']:.2f}%")
print(f"F1 Score (weighted avg): {best_info['best_f1_score']:.2f}%")
print(f"Class 1 Accuracy: {best_info['best_class_1_accuracy']:.2f}%")
print(f"Class -1 Accuracy: {best_info['best_class_minus1_accuracy']:.2f}%")


SUMMARY OF BEST K PERFORMANCE FOR EACH ALPHA:

alpha=0.0:
  Best k: 7
  Overall Accuracy: 80.31%
  F1 Score (weighted avg): 75.56%
  Class 1 Accuracy: 18.55%
  Class -1 Accuracy: 97.55%

alpha=0.1:
  Best k: 5
  Overall Accuracy: 80.09%
  F1 Score (weighted avg): 75.79%
  Class 1 Accuracy: 20.45%
  Class -1 Accuracy: 96.73%

alpha=0.2:
  Best k: 5
  Overall Accuracy: 79.88%
  F1 Score (weighted avg): 75.63%
  Class 1 Accuracy: 20.45%
  Class -1 Accuracy: 96.45%

alpha=0.3:
  Best k: 5
  Overall Accuracy: 79.88%
  F1 Score (weighted avg): 75.62%
  Class 1 Accuracy: 20.45%
  Class -1 Accuracy: 96.45%

alpha=0.4:
  Best k: 4
  Overall Accuracy: 79.25%
  F1 Score (weighted avg): 76.23%
  Class 1 Accuracy: 27.18%
  Class -1 Accuracy: 93.74%

alpha=0.5:
  Best k: 4
  Overall Accuracy: 79.68%
  F1 Score (weighted avg): 76.78%
  Class 1 Accuracy: 28.18%
  Class -1 Accuracy: 94.02%

alpha=0.6:
  Best k: 4
  Overall Accuracy: 79.47%
  F1 Score (weighted avg): 76.51%
  Class 1 Accuracy: 27.27%
 